# CELL 0 — Project Overview & Research Question

## Project title  
**Detecting Cross-Sectional Mispricing in Highly Fungible Consumer Goods on eBay**

---

## Research question  

Can we detect economically meaningful mispricing in *AirPods Pro 2* listings on eBay by:

- clustering identical items using **hard, interpretable rules**,  
- adjusting prices for **seller (counterparty) risk** and **item-level quality signals**,  
- and trading only **large residual deviations** from a robust anchor price

---

## Conceptual framing (finance analogy)

- **Item (AirPods Pro 2)** → underlying asset  
- **Seller credibility** → counterparty / credit risk  
- **Text-level defects & completeness** → idiosyncratic risk  
- **Residual price deviation** → mispricing signal  

This is **not** risk-free arbitrage.  
It is a **stat-arb–style residual pricing problem** in a two-sided marketplace.

---

## Scope & constraints (frozen)

- Product: **AirPods Pro 2 only**
- Region: **eBay US**
- Listing type: **Buy-It-Now only**
- Currency: USD
- Execution: **Paper trading only**
- No leverage, no automated buying/selling

---

## High-level pipeline

1. Ingest raw eBay listings via the official Browse API  
2. Normalize text and extract structured identity fields  
3. Apply hard filters to remove non-tradable listings  
4. Cluster comparable items using **hierarchical rule-based clustering**  
5. Estimate cluster anchor prices using **robust statistics**  
6. Adjust prices for seller risk and item-level quality  
7. Detect residual mispricing within clusters  
8. Simulate resale expected value with realistic frictions  

---

## Design principles

- Conservative by construction  
- Interpretability over complexity  
- Robust to missing or incomplete data  
- No supervised ML training required  

---

## Intended outcome

- A reproducible framework for identifying mispriced listings  
- A realistic assessment of whether mispricings survive fees and risk  
- A research-style notebook aligned with professional quantitative reasoning

In [ ]:
# CELL X — eBay OAuth (Application Token)

import time
import base64
import requests

# ====== CREDENTIALS ======
# Set EBAY_CLIENT_ID and EBAY_CLIENT_SECRET as environment variables
# (or as Colab secrets) before running. Never hardcode them.
import os
try:
    from google.colab import userdata
    EBAY_CLIENT_ID = userdata.get("EBAY_CLIENT_ID")
    EBAY_CLIENT_SECRET = userdata.get("EBAY_CLIENT_SECRET")
except ImportError:
    EBAY_CLIENT_ID = os.environ["EBAY_CLIENT_ID"]
    EBAY_CLIENT_SECRET = os.environ["EBAY_CLIENT_SECRET"]
# =========================

TOKEN_URL = "https://api.ebay.com/identity/v1/oauth2/token"
SCOPE = "https://api.ebay.com/oauth/api_scope"

_token = None
_expiry = 0

def get_ebay_token():
    global _token, _expiry

    # reuse token if still valid
    if _token and time.time() < _expiry - 60:
        return _token

    basic = base64.b64encode(
        f"{EBAY_CLIENT_ID}:{EBAY_CLIENT_SECRET}".encode()
    ).decode()

    headers = {
        "Content-Type": "application/x-www-form-urlencoded",
        "Authorization": f"Basic {basic}",
    }

    data = {
        "grant_type": "client_credentials",
        "scope": SCOPE,
    }

    r = requests.post(TOKEN_URL, headers=headers, data=data)
    r.raise_for_status()

    payload = r.json()
    _token = payload["access_token"]
    _expiry = time.time() + payload["expires_in"]

    return _token

print("Token ready.")

In [ ]:
# =========================
# CELL 1 — Environment Setup & Imports
# =========================

"""
Purpose:
- Set up the Python environment
- Import core libraries used throughout the notebook
- Verify versions for reproducibility

This cell should:
- Run without errors
- Print library versions
"""

# -------------------------
# STANDARD LIBRARIES
# -------------------------
import os
import json
from pprint import pprint
from datetime import datetime, timedelta

# -------------------------
# DATA & NUMERICS
# -------------------------
import numpy as np
import pandas as pd

# -------------------------
# HTTP / API
# -------------------------
import requests

# -------------------------
# DISPLAY / DEBUG
# -------------------------
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

# -------------------------
# VERSION CHECKS
# -------------------------
print("Python version check:")
print(os.sys.version)

print("\nLibrary versions:")
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("requests:", requests.__version__)

print("\nEnvironment setup complete.")

In [ ]:
# =========================
# EBAY API AUTH — ONE-TIME SETUP (AUTO, SAFE)
# =========================

import os

# This cell assumes get_ebay_token() is already defined ABOVE
# It preserves the existing EBAY_BEARER_TOKEN interface
# so downstream cells do NOT need to change.

def refresh_ebay_bearer_token():
    token = get_ebay_token()
    os.environ["EBAY_BEARER_TOKEN"] = token
    return token

# Initialize once
refresh_ebay_bearer_token()

print("EBAY_BEARER_TOKEN set (auto-generated, ~2h validity).")

In [ ]:
# =========================
# CELL 2 — Data Ingestion (eBay Browse API, Paginated)
# =========================

"""
Purpose:
- Call eBay Browse API
- Retrieve ALL Buy-It-Now listings for 'AirPods Pro 2' via pagination
- Inspect raw JSON structure

Success criteria:
- HTTP status code = 200 on all pages
- itemSummaries accumulated across pages
"""

import os
import requests
from pprint import pprint

# -------------------------
# CONFIG
# -------------------------
EBAY_BEARER_TOKEN = os.getenv("EBAY_BEARER_TOKEN")
if EBAY_BEARER_TOKEN is None:
    raise ValueError(
        "EBAY_BEARER_TOKEN not found. "
        "Run the auth cell first."
    )

SEARCH_QUERY = "AirPods Pro 2"
MARKETPLACE_ID = "EBAY_US"
LIMIT = 50          # keep same as before
MAX_PAGES = 50      # safety cap to avoid infinite loops

# -------------------------
# API SETUP
# -------------------------
url = "https://api.ebay.com/buy/browse/v1/item_summary/search"

headers = {
    "Authorization": f"Bearer {EBAY_BEARER_TOKEN}",
    "X-EBAY-C-MARKETPLACE-ID": MARKETPLACE_ID,
    "Content-Type": "application/json",
}

params = {
    "q": SEARCH_QUERY,
    "limit": LIMIT,
    "filter": "buyingOptions:{FIXED_PRICE}",
}

# -------------------------
# PAGINATED REQUEST LOOP
# -------------------------
all_items = []
page = 0
next_url = None

print("Starting paginated pull...")

while True:
    if next_url:
        response = requests.get(next_url, headers=headers)
    else:
        response = requests.get(url, headers=headers, params=params)

    print(f"\nPage {page + 1} HTTP status code:", response.status_code)

    if response.status_code != 200:
        print("Error response:")
        pprint(response.json())
        raise RuntimeError("eBay API call failed during pagination")

    data = response.json()
    items = data.get("itemSummaries", [])
    all_items.extend(items)

    page += 1
    print(f"Listings collected so far: {len(all_items)}")

    next_url = data.get("next")
    if not next_url:
        print("No next page. Pagination complete.")
        break

    if page >= MAX_PAGES:
        print("Reached MAX_PAGES limit. Stopping pagination.")
        break

# -------------------------
# BASIC INSPECTION (same as before)
# -------------------------
print("\nTop-level response keys (last page):")
print(list(data.keys()))

print(f"\nTotal number of listings collected: {len(all_items)}")

print("\nSample listing (first item):")
pprint(all_items[0])

In [ ]:
# =========================
# CELL 3 — Raw DataFrame Construction & Sanity Checks
# =========================

"""
Purpose:
- Convert raw eBay API response into a pandas DataFrame
- Extract only top-level fields we will use downstream
- Perform basic sanity checks (no filtering yet)

Output:
- raw_listings_df
"""

import pandas as pd

items = all_items
rows = []

for it in items:
    # --- shipping cost (robust extraction) ---
    shipping_cost = None
    if it.get("shippingOptions"):
        opt = it["shippingOptions"][0]
        if "shippingCost" in opt and "value" in opt["shippingCost"]:
            shipping_cost = float(opt["shippingCost"]["value"])
        else:
            # free or unspecified shipping
            shipping_cost = 0.0

    row = {
        # Identifiers
        "item_id": it.get("itemId"),
        "legacy_item_id": it.get("legacyItemId"),

        # Text
        "title": it.get("title"),

        # Pricing
        "price": float(it["price"]["value"]) if "price" in it else None,
        "currency": it["price"]["currency"] if "price" in it else None,

        # Shipping
        "shipping_cost": shipping_cost,

        # Condition
        "condition": it.get("condition"),
        "condition_id": it.get("conditionId"),

        # Seller info
        "seller_username": it.get("seller", {}).get("username"),
        "seller_feedback_pct": float(it["seller"]["feedbackPercentage"])
            if it.get("seller", {}).get("feedbackPercentage") else None,
        "seller_feedback_count": int(it["seller"]["feedbackScore"])
            if it.get("seller", {}).get("feedbackScore") else None,

        # Listing metadata
        "listing_date": it.get("itemCreationDate"),
        "marketplace": it.get("listingMarketplaceId"),

        # Flags
        "top_rated": it.get("topRatedBuyingExperience", False),
    }

    rows.append(row)

raw_listings_df = pd.DataFrame(rows)

# -------------------------
# BASIC SANITY CHECKS
# -------------------------
print("Number of rows:", len(raw_listings_df))
print("\nColumns:")
print(raw_listings_df.columns.tolist())

print("\nMissing values per column:")
print(raw_listings_df.isna().sum())

print("\nPrice summary:")
print(raw_listings_df["price"].describe())

raw_listings_df.head()

In [ ]:
# =========================
# CELL 4 — Text Normalization (NLP-lite)
# =========================

"""
Purpose:
- Normalize listing text into a clean, consistent form
- Prepare for rule-based extraction in CELL 5
- NO classification, NO assumptions, NO filtering

Design:
- Deterministic
- Reversible
- Conservative (keep information, remove noise)

Output:
- raw_listings_df with added normalized text columns
"""

import re

def normalize_text(text):
    if text is None:
        return None

    text = text.lower()

    # remove URLs
    text = re.sub(r"http\S+|www\S+", " ", text)

    # remove emojis / non-alphanumeric (keep basic punctuation)
    text = re.sub(r"[^a-z0-9\s\-\+\.]", " ", text)

    # normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text


# -------------------------
# APPLY NORMALIZATION
# -------------------------
raw_listings_df["title_norm"] = raw_listings_df["title"].apply(normalize_text)

# Optional placeholder for description (not yet pulled from API)
raw_listings_df["description_norm"] = None


# -------------------------
# QUICK SANITY CHECK
# -------------------------
print("Sample before / after normalization:\n")

for i in range(3):
    print(f"ORIGINAL:   {raw_listings_df.loc[i, 'title']}")
    print(f"NORMALIZED: {raw_listings_df.loc[i, 'title_norm']}")
    print("-" * 80)

raw_listings_df[["title", "title_norm"]].head()

In [ ]:
# =========================
# CELL 5 — Rule-Based Identity Extraction
# =========================

"""
Purpose:
- Extract structured identity signals from normalized title text
- Use conservative, interpretable rules
- Do NOT filter or drop any listings yet

Output:
- raw_listings_df with added identity columns
"""

import numpy as np

def extract_identity_features(title_norm):
    if title_norm is None:
        return {
            "model_confirmed": False,
            "case_type": "unknown",
            "authenticity_flag": "unknown",
            "condition_bucket": "unknown",
            "box_included": "unknown",
        }

    t = title_norm

    # -------------------------
    # Model confirmation
    # -------------------------
    model_confirmed = (
        "airpods" in t and
        "pro" in t and
        ("2nd" in t or "gen 2" in t or "2nd generation" in t)
    )

    # -------------------------
    # Case type
    # -------------------------
    if "usb-c" in t or "usbc" in t:
        case_type = "usb_c"
    elif "lightning" in t:
        case_type = "lightning"
    else:
        case_type = "unknown"

    # -------------------------
    # Authenticity flag
    # -------------------------
    if any(x in t for x in ["replica", "fake", "knockoff", "clone"]):
        authenticity_flag = "replica_suspected"
    elif any(x in t for x in ["genuine", "authentic", "original", "oem"]):
        authenticity_flag = "oem"
    else:
        authenticity_flag = "unknown"

    # -------------------------
    # Condition bucket
    # -------------------------
    if any(x in t for x in ["new", "sealed", "brand new", "open box"]):
        condition_bucket = "new_open_box"
    elif any(x in t for x in ["used", "refurb", "refurbished", "pre-owned"]):
        condition_bucket = "used_refurb"
    else:
        condition_bucket = "unknown"

    # -------------------------
    # Box included
    # -------------------------
    if any(x in t for x in ["no box", "without box", "replacement"]):
        box_included = "no"
    elif "box" in t:
        box_included = "yes"
    else:
        box_included = "unknown"

    return {
        "model_confirmed": model_confirmed,
        "case_type": case_type,
        "authenticity_flag": authenticity_flag,
        "condition_bucket": condition_bucket,
        "box_included": box_included,
    }


# -------------------------
# APPLY EXTRACTION
# -------------------------
identity_df = raw_listings_df["title_norm"].apply(
    lambda x: pd.Series(extract_identity_features(x))
)

raw_listings_df = pd.concat([raw_listings_df, identity_df], axis=1)


# -------------------------
# QUICK SANITY CHECKS
# -------------------------
print("Model confirmed counts:")
print(raw_listings_df["model_confirmed"].value_counts())

print("\nCase type counts:")
print(raw_listings_df["case_type"].value_counts())

print("\nAuthenticity flag counts:")
print(raw_listings_df["authenticity_flag"].value_counts())

print("\nCondition bucket counts:")
print(raw_listings_df["condition_bucket"].value_counts())

print("\nBox included counts:")
print(raw_listings_df["box_included"].value_counts())

raw_listings_df.head()

In [ ]:
# =========================
# CELL 6 — Hard Filters (Trash Removal)
# =========================

"""
Purpose:
- Remove clearly non-tradable listings
- Apply irreversible filters conservatively
- Keep only plausible AirPods Pro 2 listings

Output:
- tradable_listings_df
"""

# Start from raw listings
df = raw_listings_df.copy()

initial_count = len(df)

# -------------------------
# HARD FILTERS
# -------------------------

# 1. Must look like AirPods Pro 2
df = df[df["model_confirmed"] == True]

# 2. Drop explicit replicas
df = df[df["authenticity_flag"] != "replica_suspected"]

# 3. Drop replacement-only items
def is_replacement_only(title_norm):
    if title_norm is None:
        return False

    # hard replacement signals
    hard_signals = [
        "replacement only",
        "case only",
        "left only",
        "right only",
        "earbud only",
        "charging case only"
    ]

    # soft signals that require context
    soft_signals = [
        "replacement",
        "charging case"
    ]

    # If hard signal appears → drop
    if any(h in title_norm for h in hard_signals):
        return True

    # If soft signal appears, but NO "with" or plural context → drop
    if any(s in title_norm for s in soft_signals):
        if "with" not in title_norm and "earbuds" not in title_norm:
            return True

    return False

df = df[~df["title_norm"].apply(is_replacement_only)]

# 4. Drop unrealistically cheap listings (economic identity guardrail)
PRICE_FLOOR = 70.0
df = df[df["price"] >= PRICE_FLOOR]

# -------------------------
# RESULTS
# -------------------------
final_count = len(df)

print(f"Initial listings: {initial_count}")
print(f"Remaining after hard filters: {final_count}")
print(f"Removed: {initial_count - final_count}")

tradable_listings_df = df.reset_index(drop=True)

tradable_listings_df.head()

In [ ]:
# =========================
# CELL 7 — Item Quality Risk Flags (Soft Penalties)
# =========================

"""
Purpose:
- Identify item-level quality issues from text
- These do NOT filter listings
- They will later reduce fair value estimates

Design:
- Conservative
- Deterministic
- No dropping rows

Output:
- tradable_listings_df with added quality-flag columns
"""

import pandas as pd

def extract_quality_flags(title_norm):
    if title_norm is None:
        return {
            "cosmetic_damage": False,
            "heavy_usage": False,
            "missing_accessories": False,
            "engraved": False,
        }

    t = title_norm

    cosmetic_damage = any(x in t for x in [
        "scratch", "scratches", "scuff", "crack", "dent"
    ])

    heavy_usage = any(x in t for x in [
        "used heavily", "well used", "worn", "signs of use"
    ])

    missing_accessories = any(x in t for x in [
        "no tips", "missing tips", "no ear tips",
        "no cable", "no charger"
    ])

    engraved = any(x in t for x in [
        "engraved", "engraving"
    ])

    return {
        "cosmetic_damage": cosmetic_damage,
        "heavy_usage": heavy_usage,
        "missing_accessories": missing_accessories,
        "engraved": engraved,
    }


# -------------------------
# APPLY QUALITY FLAGS
# -------------------------
quality_df = tradable_listings_df["title_norm"].apply(
    lambda x: pd.Series(extract_quality_flags(x))
)

tradable_listings_df = pd.concat(
    [tradable_listings_df, quality_df],
    axis=1
)

# -------------------------
# SANITY CHECK
# -------------------------
print("Quality flag counts:")
print(
    tradable_listings_df[
        ["cosmetic_damage", "heavy_usage", "missing_accessories", "engraved"]
    ].sum()
)

tradable_listings_df.head()

In [ ]:
# =========================
# CELL 8 — Seller Risk Signals (Counterparty Risk)
# =========================

"""
Purpose:
- Extract seller-side risk indicators
- These do NOT filter listings
- They will later reduce fair value estimates

Design:
- Conservative
- Interpretable
- No ML, no fitting

Output:
- tradable_listings_df with added seller-risk columns
"""

def extract_seller_risk(row):
    pct = row["seller_feedback_pct"]
    cnt = row["seller_feedback_count"]
    top_rated = row["top_rated"]

    # -------------------------
    # Feedback percentage risk
    # -------------------------
    very_low_feedback_pct = pct < 98.5
    low_feedback_pct = (pct >= 98.5) and (pct < 99.5)

    # -------------------------
    # Feedback count risk
    # -------------------------
    low_feedback_count = cnt < 100
    very_low_feedback_count = cnt < 20

    # -------------------------
    # Top-rated penalty
    # -------------------------
    not_top_rated = not bool(top_rated)

    # -------------------------
    # Simple additive risk score
    # (used later as price penalty weight)
    # -------------------------
    seller_risk_score = 0

    if very_low_feedback_pct:
        seller_risk_score += 2
    elif low_feedback_pct:
        seller_risk_score += 1

    if very_low_feedback_count:
        seller_risk_score += 2
    elif low_feedback_count:
        seller_risk_score += 1

    if not_top_rated:
        seller_risk_score += 1

    return {
        "low_feedback_pct": low_feedback_pct,
        "very_low_feedback_pct": very_low_feedback_pct,
        "low_feedback_count": low_feedback_count,
        "very_low_feedback_count": very_low_feedback_count,
        "not_top_rated": not_top_rated,
        "seller_risk_score": seller_risk_score,
    }


# -------------------------
# APPLY SELLER RISK EXTRACTION
# -------------------------
seller_risk_df = tradable_listings_df.apply(
    extract_seller_risk,
    axis=1,
    result_type="expand"
)

tradable_listings_df = pd.concat(
    [tradable_listings_df, seller_risk_df],
    axis=1
)

# -------------------------
# SANITY CHECKS
# -------------------------
print("Seller risk score distribution:")
print(tradable_listings_df["seller_risk_score"].value_counts().sort_index())

print("\nLow feedback pct count:", tradable_listings_df["low_feedback_pct"].sum())
print("Very low feedback pct count:", tradable_listings_df["very_low_feedback_pct"].sum())
print("Low feedback count:", tradable_listings_df["low_feedback_count"].sum())
print("Very low feedback count:", tradable_listings_df["very_low_feedback_count"].sum())
print("Not top rated:", tradable_listings_df["not_top_rated"].sum())

tradable_listings_df.head()

In [ ]:
# =========================
# CELL 9 — Hard-Rule Clustering (Item Identity)
# =========================

"""
Purpose:
- Cluster listings into semantically comparable groups
- Use ONLY item identity features
- No price, no seller risk, no quality flags

Output:
- tradable_listings_df with cluster_id column
"""

# -------------------------
# DEFINE CLUSTER LABEL
# -------------------------
def assign_cluster(row):
    case = row["case_type"]
    cond = row["condition_bucket"]

    return f"case={case}__cond={cond}"


tradable_listings_df["cluster_id"] = tradable_listings_df.apply(
    assign_cluster,
    axis=1
)

# -------------------------
# CLUSTER DISTRIBUTION
# -------------------------
print("Cluster sizes:")
print(tradable_listings_df["cluster_id"].value_counts())

# -------------------------
# QUICK INSPECTION
# -------------------------
tradable_listings_df[
    ["title", "price", "case_type", "condition_bucket", "cluster_id"]
].sort_values("cluster_id").head(10)

In [ ]:
# =========================
# CELL 10 — Cluster Price Anchors (Median + MAD)
# =========================

"""
Purpose:
- Compute robust price anchors per cluster
- Handle small clusters explicitly
- NO seller risk
- NO item quality adjustments

Output:
- cluster_anchor_df
- tradable_listings_df with anchor_price + anchor_dispersion
"""

import numpy as np

MIN_CLUSTER_SIZE = 3

# -------------------------
# CLUSTER SIZES
# -------------------------
cluster_sizes = tradable_listings_df["cluster_id"].value_counts()

# Identify valid clusters
valid_clusters = cluster_sizes[cluster_sizes >= MIN_CLUSTER_SIZE].index.tolist()
small_clusters = cluster_sizes[cluster_sizes < MIN_CLUSTER_SIZE].index.tolist()

print("Valid clusters (used for anchors):")
print(valid_clusters)

print("\nSmall clusters (will be rolled up):")
print(small_clusters)


# -------------------------
# FALLBACK CLUSTER MAPPING
# -------------------------
def fallback_cluster(cluster_id):
    # Simple, interpretable roll-ups
    if "usb_c" in cluster_id:
        return "case=usb_c__cond=unknown"
    else:
        return "case=unknown__cond=unknown"


tradable_listings_df["effective_cluster"] = tradable_listings_df["cluster_id"].apply(
    lambda c: c if c in valid_clusters else fallback_cluster(c)
)


# -------------------------
# COMPUTE ANCHORS
# -------------------------
anchor_rows = []

for cluster, grp in tradable_listings_df.groupby("effective_cluster"):
    prices = grp["price"].values

    median_price = np.median(prices)
    mad = np.median(np.abs(prices - median_price))

    anchor_rows.append({
        "effective_cluster": cluster,
        "anchor_price": median_price,
        "anchor_mad": mad,
        "n_items": len(prices),
    })

cluster_anchor_df = pd.DataFrame(anchor_rows)

print("\nCluster anchors:")
cluster_anchor_df


# -------------------------
# MERGE BACK TO ITEMS
# -------------------------
tradable_listings_df = tradable_listings_df.merge(
    cluster_anchor_df,
    on="effective_cluster",
    how="left"
)

tradable_listings_df[
    ["title", "price", "effective_cluster", "anchor_price", "anchor_mad"]
].head()

In [ ]:
# =========================
# CELL 11 — Fair Price Estimation (Robust Log-Linear, Centered)
# =========================

"""
Purpose:
- Estimate fair price using a statistically grounded model
- Symmetric treatment: penalize worse-than-average AND reward better-than-average
- No hand-tuned penalties

Model:
log(price) = cluster_intercept + beta * (features − cluster_mean) + noise
"""

import numpy as np
import statsmodels.api as sm

tradable_listings_df["fair_price"] = np.nan

# -------------------------
# FEATURES USED IN MODEL
# -------------------------
FEATURE_COLS = [
    "seller_risk_score",
    "cosmetic_damage",
    "heavy_usage",
    "missing_accessories",
    "engraved",
]

# Ensure numeric
for col in FEATURE_COLS:
    tradable_listings_df[col] = tradable_listings_df[col].astype(int)

# -------------------------
# FIT MODEL PER CLUSTER
# -------------------------
for cluster, grp in tradable_listings_df.groupby("effective_cluster"):
    if len(grp) < 10:
        continue  # skip tiny clusters safely

    # Target
    y = np.log(grp["price"].values)

    # --- CRITICAL FIX: center features within cluster ---
    X_raw = grp[FEATURE_COLS]
    X = X_raw - X_raw.mean()

    # Add intercept
    X = sm.add_constant(X)

    # Robust regression
    model = sm.RLM(y, X, M=sm.robust.norms.HuberT())
    res = model.fit()

    # Predict fair log-price
    log_fair = res.predict(X)

    # Back to price space
    fair_price = np.exp(log_fair)

    tradable_listings_df.loc[grp.index, "fair_price"] = fair_price

# -------------------------
# SANITY CHECK
# -------------------------
tradable_listings_df[
    ["title", "price", "fair_price", "effective_cluster"]
].dropna().head(10)

In [ ]:
# =========================
# CELL 12 — In-Cluster Arbitrage Signal
# =========================

"""
Purpose:
- Convert raw residuals into a clean arbitrage score
- Compare listings ONLY within their cluster
- Remove scale effects (price level, volatility)

Definition:
residual = price - fair_price

arb_score = residual / cluster_residual_std

Interpretation:
- arb_score << 0  → strong underpricing
- arb_score ≈ 0   → fairly priced
- arb_score >> 0  → overpriced
"""

import numpy as np

# -------------------------
# COMPUTE RAW RESIDUAL
# -------------------------
tradable_listings_df["residual"] = (
    tradable_listings_df["price"] - tradable_listings_df["fair_price"]
)

# -------------------------
# NORMALIZE WITHIN CLUSTER
# -------------------------
arb_scores = []

for cluster, grp in tradable_listings_df.groupby("effective_cluster"):
    if grp["residual"].isna().all():
        continue

    std = grp["residual"].std()

    # Avoid division blowups
    if std == 0 or np.isnan(std):
        tradable_listings_df.loc[grp.index, "arb_score"] = 0.0
    else:
        tradable_listings_df.loc[grp.index, "arb_score"] = (
            grp["residual"] / std
        )

# -------------------------
# SANITY CHECK
# -------------------------
print("Arbitrage score summary:")
print(tradable_listings_df["arb_score"].describe())

print("\nTop underpriced candidates (within cluster):")
tradable_listings_df[
    [
        "title",
        "price",
        "fair_price",
        "residual",
        "arb_score",
        "effective_cluster",
    ]
].sort_values("arb_score").head(10)

In [ ]:
# =========================
# CELL 13 — Arbitrage Ranking (Binary Seller Gate)
# =========================

"""
Purpose:
- Rank listings by pure economic mispricing (within cluster)
- Apply seller trustworthiness as a HARD execution gate
- Produce a clean arbitrage shortlist

Key rule:
- Seller risk does NOT affect score
- Seller risk only determines executability
"""

import numpy as np

df = tradable_listings_df.copy()

# -------------------------
# PURE ECONOMIC SIGNAL
# -------------------------
# Normalize residuals within cluster
df["arb_score"] = (
    df["residual"]
    - df.groupby("effective_cluster")["residual"].transform("mean")
) / (
    df.groupby("effective_cluster")["residual"].transform("std")
)

# More negative = more underpriced
# (keep sign intuitive)
# -------------------------

# -------------------------
# BINARY SELLER EXECUTION GATE
# -------------------------
# You can tune this, but keep it SIMPLE
MAX_ACCEPTABLE_SELLER_RISK = 2

df["executable"] = df["seller_risk_score"] <= MAX_ACCEPTABLE_SELLER_RISK

# -------------------------
# FINAL RANKED CANDIDATES
# -------------------------
ranked_candidates_df = (
    df[df["executable"]]
    .sort_values("arb_score")
    .reset_index(drop=True)
)

# -------------------------
# SANITY CHECKS
# -------------------------
print("Arbitrage score summary (executable only):")
print(ranked_candidates_df["arb_score"].describe())

print("\nTop executable arbitrage candidates:")
ranked_candidates_df[
    [
        "title",
        "price",
        "fair_price",
        "residual",
        "arb_score",
        "seller_risk_score",
        "effective_cluster",
    ]
].head(15)

In [ ]:
# =========================
# CELL 14 — Buy Signal Selection (Thresholded Arbitrage)
# =========================

"""
Purpose:
- Convert continuous arbitrage scores into discrete buy signals
- Focus only on statistically meaningful mispricing
- Keep logic interpretable and conservative

Key idea:
- Arbitrage lives in the LEFT TAIL
- We buy only extreme negative z-scores
"""

# -------------------------
# BUY THRESHOLD (tunable)
# -------------------------
BUY_Z_THRESHOLD = -1.0
# Try: -1.0 (aggressive), -1.5 (reasonable), -2.0 (very selective)

# -------------------------
# SELECT BUY CANDIDATES
# -------------------------
buy_candidates_df = ranked_candidates_df[
    ranked_candidates_df["arb_score"] <= BUY_Z_THRESHOLD
].reset_index(drop=True)

# -------------------------
# SANITY CHECKS
# -------------------------
print(f"Buy threshold (arb_score ≤ {BUY_Z_THRESHOLD})")
print(f"Number of buy candidates: {len(buy_candidates_df)}")

print("\nBuy candidates preview:")
buy_candidates_df[
    [
        "title",
        "price",
        "fair_price",
        "residual",
        "arb_score",
        "seller_risk_score",
        "effective_cluster",
    ]
].head(50)